In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
REPO_DIR        = "/workspace/APGCC"
CONFIG_PATH     = f"{REPO_DIR}/apgcc/configs/steelbar_train.yml"
CHECKPOINT_PATH = f"{REPO_DIR}/apgcc/output/steelbar/best.pth"
IMAGE_PATH      = f"/workspace/00000666768100048441.Image.001404.jpg"                   # <-- set your image path
OUTPUT_IMAGE    = f"/workspace/inference_result.jpg"
OUTPUT_JSON     = f"/workspace/inference_result.json"

THRESHOLD     = 0.5
SLICE_SIZE    = 512
SLICE_OVERLAP = 128
DEDUP_DIST    = 20


import json
from pathlib import Path

import cv2
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt

APGCC_DIR = str(Path(REPO_DIR) / "apgcc")
if APGCC_DIR not in sys.path:
    sys.path.insert(0, APGCC_DIR)

from models import build_model
from models.APGCC import NestedTensor
from config import cfg, merge_from_file

# ── Verify paths exist ─────────────────────────────────────────────────────
for p in [CONFIG_PATH, CHECKPOINT_PATH, IMAGE_PATH]:
    status = "OK" if Path(p).exists() else "NOT FOUND"
    print(f"[{status}] {p}")

# ── Load model ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

cfg_loaded = merge_from_file(cfg, CONFIG_PATH)
model = build_model(cfg=cfg_loaded, training=False)
model.to(device)

state = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
if isinstance(state, dict) and "model" in state:
    state = state["model"]
state = {k.replace("module.", "", 1): v for k, v in state.items()}
model.load_state_dict(state, strict=True)
model.eval()
print(f"Model loaded — {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params")

# ── Inference helpers ──────────────────────────────────────────────────────
def preprocess_pil(image):
    tensor = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])(image).unsqueeze(0)
    mask = torch.zeros((1, tensor.shape[2], tensor.shape[3]), dtype=torch.bool)
    return NestedTensor(tensor, mask)

def generate_slices(image_w, image_h, slice_size, overlap):
    stride = slice_size - overlap
    for y0 in range(0, image_h, stride):
        for x0 in range(0, image_w, stride):
            x1 = min(x0 + slice_size, image_w)
            y1 = min(y0 + slice_size, image_h)
            yield max(0, x1 - slice_size), max(0, y1 - slice_size), x1, y1

def deduplicate_points(points, scores, min_dist):
    if len(points) == 0:
        return points, scores
    order = np.argsort(-scores)
    points, scores = points[order], scores[order]
    keep = np.ones(len(points), dtype=bool)
    for i in range(len(points)):
        if not keep[i]:
            continue
        dx = points[i, 0] - points[i + 1:, 0]
        dy = points[i, 1] - points[i + 1:, 1]
        keep[i + 1:][dx * dx + dy * dy < min_dist * min_dist] = False
    return points[keep], scores[keep]

@torch.no_grad()
def predict_slice(nested):
    outputs = model(nested.to(device))
    scores = torch.softmax(outputs["pred_logits"], dim=-1)[0, :, 1]
    points = outputs["pred_points"][0]
    keep = scores > THRESHOLD
    return points[keep].cpu().numpy(), scores[keep].cpu().numpy()

# ── Run sliding-window inference ───────────────────────────────────────────
image = Image.open(IMAGE_PATH).convert("RGB")
image_w, image_h = image.size
slices = list(generate_slices(image_w, image_h, SLICE_SIZE, SLICE_OVERLAP))
print(f"Image: {image_w}x{image_h} — {len(slices)} slices")

all_points, all_scores = [], []
for idx, (x0, y0, x1, y1) in enumerate(slices):
    pts, scs = predict_slice(preprocess_pil(image.crop((x0, y0, x1, y1))))
    if len(pts):
        pts[:, 0] += x0;  pts[:, 1] += y0
        all_points.append(pts);  all_scores.append(scs)
    if (idx + 1) % 20 == 0 or (idx + 1) == len(slices):
        print(f"  {idx + 1}/{len(slices)} slices…")

if all_points:
    all_points = np.concatenate(all_points)
    all_scores = np.concatenate(all_scores)
    all_points, all_scores = deduplicate_points(all_points, all_scores, DEDUP_DIST)
else:
    all_points = np.empty((0, 2), np.float32)
    all_scores = np.empty((0,), np.float32)

print(f"\nPredicted count: {len(all_points)}")

# ── Draw & save ────────────────────────────────────────────────────────────
img_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
for (x, y), s in zip(all_points.astype(int), all_scores):
    cv2.circle(img_bgr, (x, y), 4, (0, 0, 255), -1)
    cv2.putText(img_bgr, f"{s:.2f}", (x + 6, y - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
label = f"Count: {len(all_points)}"
(tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
cv2.rectangle(img_bgr, (10, 10), (20 + tw, 20 + th), (0, 0, 0), -1)
cv2.putText(img_bgr, label, (15, 15 + th),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

cv2.imwrite(OUTPUT_IMAGE, img_bgr)

Path(OUTPUT_JSON).write_text(json.dumps({
    "count": int(len(all_points)),
    "threshold": THRESHOLD,
    "slice_size": SLICE_SIZE,
    "slice_overlap": SLICE_OVERLAP,
    "points": [{"x": float(p[0]), "y": float(p[1]), "score": float(s)}
               for p, s in zip(all_points, all_scores)],
}, indent=2), encoding="utf-8")

print(f"Saved image : {OUTPUT_IMAGE}")
print(f"Saved JSON  : {OUTPUT_JSON}")

# ── Display ────────────────────────────────────────────────────────────────
plt.figure(figsize=(16, 10))
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title(f"APGCC — Count: {len(all_points)}  (threshold={THRESHOLD})", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.show()